# Feature Engineering

Build a leakage-free chronological Player 1 vs Player 2 dataset.

In [2]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = (
    PROJECT_ROOT
    / "tennis-sackmann-archive-main-aneeshers"
    / "tennis-sackmann-archive-main"
    / "atp"
)

DATA_DIR.exists()

True

In [4]:
yearly_frames = []

for year in range(1981, 2027):
    yearly_matches = pd.read_csv(
        DATA_DIR / f"atp_matches_{year}.csv"
    )
    yearly_matches["source_year"] = year
    yearly_frames.append(yearly_matches)

matches = pd.concat(
    yearly_frames,
    ignore_index=True,
)

matches["tourney_date"] = pd.to_datetime(
    matches["tourney_date"].astype(str),
    format="%Y%m%d",
)

In [5]:
matches.shape, matches["tourney_date"].agg(["min", "max"])

((148669, 50),
 min   1981-01-05
 max   2026-05-25
 Name: tourney_date, dtype: datetime64[us])

In [6]:
state_matches = matches.loc[
    matches["tourney_level"].isin(["A", "M", "G", "F"])
].copy()

score_upper = (
    state_matches["score"]
    .fillna("")
    .str.upper()
)

state_matches["match_status"] = "completed"

state_matches.loc[
    score_upper.eq(""),
    "match_status",
] = "missing_score"

state_matches.loc[
    score_upper.str.contains(r"W/O|WALKOVER", regex=True),
    "match_status",
] = "walkover"

state_matches.loc[
    score_upper.str.contains(r"\bRET\b", regex=True),
    "match_status",
] = "retirement"

state_matches.loc[
    score_upper.str.contains(r"\bDEF\b", regex=True),
    "match_status",
] = "default"

state_matches.loc[
    score_upper.str.contains(
        r"ABD|ABN|UNFINISHED|IN PROGRESS",
        regex=True,
    ),
    "match_status",
] = "abandoned"

In [7]:
state_matches["is_model_eligible"] = (
    state_matches["surface"].isin(["Hard", "Clay", "Grass"])
    & state_matches["match_status"].eq("completed")
)

In [8]:
(
    state_matches.shape,
    state_matches["is_model_eligible"].sum(),
    state_matches["match_status"].value_counts(),
)

((135868, 52),
 np.int64(120917),
 match_status
 completed        132069
 retirement         3142
 walkover            596
 default              58
 missing_score         3
 Name: count, dtype: int64)

In [9]:
round_order_map = {
    "ER": 0,
    "RR": 0,
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "BR": 7,
    "F": 8,
}

state_matches["round_order"] = (
    state_matches["round"]
    .map(round_order_map)
)

state_matches["source_row_order"] = state_matches.index

In [10]:
state_matches = (
    state_matches
    .sort_values(
        [
            "tourney_date",
            "tourney_id",
            "round_order",
            "match_num",
            "source_row_order",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

In [11]:
(
    state_matches["round_order"].isna().sum(),
    state_matches["tourney_date"].is_monotonic_increasing,
)

(np.int64(0), True)

In [12]:
def assign_winner_to_p1(tourney_id, match_num):
    match_key = f"{tourney_id}|{match_num}".encode("utf-8")
    digest = hashlib.sha256(match_key).digest()

    return digest[0] % 2 == 0

In [13]:
state_matches["winner_is_p1"] = [
    assign_winner_to_p1(tourney_id, match_num)
    for tourney_id, match_num in zip(
        state_matches["tourney_id"],
        state_matches["match_num"],
    )
]

state_matches["target"] = (
    state_matches["winner_is_p1"]
    .astype("int8")
)

In [14]:
state_matches.loc[
    state_matches["is_model_eligible"],
    "target",
].value_counts(normalize=True).sort_index()

target
0    0.498764
1    0.501236
Name: proportion, dtype: float64

In [15]:
player_fields = [
    "id",
    "seed",
    "entry",
    "name",
    "hand",
    "ht",
    "ioc",
    "age",
    "rank",
    "rank_points",
]

for field in player_fields:
    winner_column = f"winner_{field}"
    loser_column = f"loser_{field}"

    state_matches[f"p1_{field}"] = np.where(
        state_matches["winner_is_p1"],
        state_matches[winner_column],
        state_matches[loser_column],
    )

    state_matches[f"p2_{field}"] = np.where(
        state_matches["winner_is_p1"],
        state_matches[loser_column],
        state_matches[winner_column],
    )

In [16]:
assert (
    state_matches.loc[
        state_matches["target"].eq(1),
        "p1_id",
    ]
    == state_matches.loc[
        state_matches["target"].eq(1),
        "winner_id",
    ]
).all()

assert (
    state_matches.loc[
        state_matches["target"].eq(0),
        "p2_id",
    ]
    == state_matches.loc[
        state_matches["target"].eq(0),
        "winner_id",
    ]
).all()

print("P1/P2 mapping correct")

P1/P2 mapping correct


In [17]:
state_matches["rank_difference"] = (
    state_matches["p2_rank"]
    - state_matches["p1_rank"]
)

state_matches["rank_points_difference"] = (
    state_matches["p1_rank_points"]
    - state_matches["p2_rank_points"]
)

state_matches["age_difference"] = (
    state_matches["p1_age"]
    - state_matches["p2_age"]
)

state_matches["height_difference"] = (
    state_matches["p1_ht"]
    - state_matches["p2_ht"]
)

In [18]:
state_matches["p1_is_seeded"] = (
    state_matches["p1_seed"].notna()
)

state_matches["p2_is_seeded"] = (
    state_matches["p2_seed"].notna()
)

state_matches["seed_advantage"] = (
    state_matches["p2_seed"]
    - state_matches["p1_seed"]
)

state_matches["hand_matchup"] = (
    state_matches["p1_hand"].fillna("U").astype(str)
    + "_"
    + state_matches["p2_hand"].fillna("U").astype(str)
)

In [19]:
state_matches.loc[
    state_matches["is_model_eligible"],
    [
        "p1_name",
        "p2_name",
        "target",
        "rank_difference",
        "rank_points_difference",
        "age_difference",
        "height_difference",
        "hand_matchup",
    ],
].tail()

,p1_name,p2_name,target,rank_difference,rank_points_difference,age_difference,height_difference,hand_matchup
135862,Flavio Cobolli,Felix Auger Aliassime,1,-8.0,-1710.0,-1.7,-10.0,R_R
135863,Joao Fonseca,Jakub Mensik,0,-3.0,-115.0,-1.0,-8.0,R_R
135864,Rafael Jodar,Alexander Zverev,0,-26.0,-4244.0,-9.4,NaN,U_R
135866,Jakub Mensik,Alexander Zverev,0,-24.0,-4155.0,-8.3,-5.0,R_R
135867,Flavio Cobolli,Alexander Zverev,0,-11.0,-3365.0,-5.0,-15.0,R_R


In [20]:
INITIAL_ELO = 1500.0
ELO_DIVISOR = 400.0

PROVISIONAL_MATCH_LIMIT = 10
PROVISIONAL_K = 80.0
STANDARD_K = 40.0

In [21]:
def get_k_factor(matches_played):
    if matches_played < PROVISIONAL_MATCH_LIMIT:
        return PROVISIONAL_K

    return STANDARD_K

In [24]:
def calculate_elo_expected_score(rating_a, rating_b):
    rating_difference = rating_b - rating_a

    return 1.0 / (
        1.0
        + 10.0 ** (rating_difference / ELO_DIVISOR)
    )

In [25]:
assert np.isclose(
    calculate_elo_expected_score(1500, 1500),
    0.5,
)

assert np.isclose(
    calculate_elo_expected_score(1900, 1500),
    0.9090909,
)

assert np.isclose(
    calculate_elo_expected_score(1500, 1900),
    0.0909091,
)

print("Elo helpers correct")

Elo helpers correct


In [26]:
def update_elo(
    winner_rating,
    loser_rating,
    winner_matches_played,
    loser_matches_played,
):
    winner_expected_score = calculate_elo_expected_score(
        winner_rating,
        loser_rating,
    )

    loser_expected_score = 1.0 - winner_expected_score

    winner_k = get_k_factor(winner_matches_played)
    loser_k = get_k_factor(loser_matches_played)

    winner_elo_change = (
        winner_k
        * (1.0 - winner_expected_score)
    )

    loser_elo_change = (
        loser_k
        * (0.0 - loser_expected_score)
    )

    new_winner_rating = (
        winner_rating
        + winner_elo_change
    )

    new_loser_rating = (
        loser_rating
        + loser_elo_change
    )

    return (
        new_winner_rating,
        new_loser_rating,
        winner_elo_change,
        loser_elo_change,
    )

In [27]:
result = update_elo(
    winner_rating=1500,
    loser_rating=1500,
    winner_matches_played=0,
    loser_matches_played=0,
)

result

(1540.0, 1460.0, 40.0, -40.0)

In [28]:
result = update_elo(
    winner_rating=1500,
    loser_rating=1500,
    winner_matches_played=0,
    loser_matches_played=20,
)

result

(1540.0, 1480.0, 40.0, -20.0)

In [29]:
general_ratings = {}
general_match_counts = {}

winner_elo_before = []
loser_elo_before = []

winner_matches_before = []
loser_matches_before = []

postmatch_winner_elo_change = []
postmatch_loser_elo_change = []

In [30]:
for match in state_matches.itertuples(index=False):
    winner_id = int(match.winner_id)
    loser_id = int(match.loser_id)

    winner_rating = general_ratings.get(
        winner_id,
        INITIAL_ELO,
    )

    loser_rating = general_ratings.get(
        loser_id,
        INITIAL_ELO,
    )

    winner_match_count = general_match_counts.get(
        winner_id,
        0,
    )

    loser_match_count = general_match_counts.get(
        loser_id,
        0,
    )

    # Feature değerlerini update'ten önce kaydet.
    winner_elo_before.append(winner_rating)
    loser_elo_before.append(loser_rating)

    winner_matches_before.append(winner_match_count)
    loser_matches_before.append(loser_match_count)

    if match.match_status == "completed":
        (
            new_winner_rating,
            new_loser_rating,
            winner_elo_change,
            loser_elo_change,
        ) = update_elo(
            winner_rating=winner_rating,
            loser_rating=loser_rating,
            winner_matches_played=winner_match_count,
            loser_matches_played=loser_match_count,
        )

        general_ratings[winner_id] = new_winner_rating
        general_ratings[loser_id] = new_loser_rating

        general_match_counts[winner_id] = (
            winner_match_count + 1
        )

        general_match_counts[loser_id] = (
            loser_match_count + 1
        )

        postmatch_winner_elo_change.append(
            winner_elo_change
        )

        postmatch_loser_elo_change.append(
            loser_elo_change
        )

    else:
        postmatch_winner_elo_change.append(np.nan)
        postmatch_loser_elo_change.append(np.nan)

In [31]:
state_matches["winner_elo_before"] = winner_elo_before
state_matches["loser_elo_before"] = loser_elo_before

state_matches["winner_matches_before"] = (
    winner_matches_before
)

state_matches["loser_matches_before"] = (
    loser_matches_before
)

state_matches["postmatch_winner_elo_change"] = (
    postmatch_winner_elo_change
)

state_matches["postmatch_loser_elo_change"] = (
    postmatch_loser_elo_change
)

In [32]:
state_matches["p1_elo"] = np.where(
    state_matches["winner_is_p1"],
    state_matches["winner_elo_before"],
    state_matches["loser_elo_before"],
)

state_matches["p2_elo"] = np.where(
    state_matches["winner_is_p1"],
    state_matches["loser_elo_before"],
    state_matches["winner_elo_before"],
)

state_matches["p1_matches_played"] = np.where(
    state_matches["winner_is_p1"],
    state_matches["winner_matches_before"],
    state_matches["loser_matches_before"],
)

state_matches["p2_matches_played"] = np.where(
    state_matches["winner_is_p1"],
    state_matches["loser_matches_before"],
    state_matches["winner_matches_before"],
)

In [33]:
state_matches["elo_difference"] = (
    state_matches["p1_elo"]
    - state_matches["p2_elo"]
)

state_matches["elo_total"] = (
    state_matches["p1_elo"]
    + state_matches["p2_elo"]
)

state_matches["experience_difference"] = (
    state_matches["p1_matches_played"]
    - state_matches["p2_matches_played"]
)

In [ ]:
completed_match_count = (
    state_matches["match_status"]
    .eq("completed")
    .sum()
)

assert sum(general_match_counts.values()) == (
    2 * completed_match_countS
)

assert state_matches.loc[
    state_matches["match_status"].ne("completed"),
    "postmatch_winner_elo_change",
].isna().all()

print("General Elo state engine correct")

General Elo state engine correct


In [35]:
winner_events = pd.DataFrame({
    "event_order": state_matches.index,
    "player_id": state_matches["winner_id"],
    "elo_before": state_matches["winner_elo_before"],
    "elo_change": state_matches[
        "postmatch_winner_elo_change"
    ],
    "matches_before": state_matches[
        "winner_matches_before"
    ],
    "completed": state_matches[
        "match_status"
    ].eq("completed").astype(int),
})

loser_events = pd.DataFrame({
    "event_order": state_matches.index,
    "player_id": state_matches["loser_id"],
    "elo_before": state_matches["loser_elo_before"],
    "elo_change": state_matches[
        "postmatch_loser_elo_change"
    ],
    "matches_before": state_matches[
        "loser_matches_before"
    ],
    "completed": state_matches[
        "match_status"
    ].eq("completed").astype(int),
})

In [36]:
player_events = (
    pd.concat(
        [winner_events, loser_events],
        ignore_index=True,
    )
    .sort_values(
        ["event_order", "player_id"],
        kind="stable",
    )
)

In [37]:
player_events["expected_next_elo"] = (
    player_events["elo_before"]
    + player_events["elo_change"].fillna(0)
)

player_events["actual_next_elo"] = (
    player_events
    .groupby("player_id")["elo_before"]
    .shift(-1)
)

player_events["expected_next_match_count"] = (
    player_events["matches_before"]
    + player_events["completed"]
)

player_events["actual_next_match_count"] = (
    player_events
    .groupby("player_id")["matches_before"]
    .shift(-1)
)

In [38]:
has_next_match = (
    player_events["actual_next_elo"].notna()
)

assert np.allclose(
    player_events.loc[
        has_next_match,
        "expected_next_elo",
    ],
    player_events.loc[
        has_next_match,
        "actual_next_elo",
    ],
)

assert np.array_equal(
    player_events.loc[
        has_next_match,
        "expected_next_match_count",
    ].to_numpy(),
    player_events.loc[
        has_next_match,
        "actual_next_match_count",
    ].to_numpy(),
)

first_player_events = (
    player_events
    .groupby("player_id", sort=False)
    .head(1)
)

assert np.allclose(
    first_player_events["elo_before"],
    INITIAL_ELO,
)

assert (
    first_player_events["matches_before"] == 0
).all()

print("General Elo is chronological and leakage-free")

General Elo is chronological and leakage-free


In [39]:
SUPPORTED_SURFACES = ("Hard", "Clay", "Grass")

surface_ratings = {
    surface: {}
    for surface in SUPPORTED_SURFACES
}

surface_match_counts = {
    surface: {}
    for surface in SUPPORTED_SURFACES
}

In [40]:
winner_surface_elo_before = []
loser_surface_elo_before = []

winner_surface_matches_before = []
loser_surface_matches_before = []

postmatch_winner_surface_elo_change = []
postmatch_loser_surface_elo_change = []

In [41]:
for match in state_matches.itertuples(index=False):
    surface = match.surface

    if surface not in SUPPORTED_SURFACES:
        winner_surface_elo_before.append(np.nan)
        loser_surface_elo_before.append(np.nan)

        winner_surface_matches_before.append(np.nan)
        loser_surface_matches_before.append(np.nan)

        postmatch_winner_surface_elo_change.append(np.nan)
        postmatch_loser_surface_elo_change.append(np.nan)

        continue

    winner_id = int(match.winner_id)
    loser_id = int(match.loser_id)

    ratings = surface_ratings[surface]
    match_counts = surface_match_counts[surface]

    winner_rating = ratings.get(
        winner_id,
        INITIAL_ELO,
    )

    loser_rating = ratings.get(
        loser_id,
        INITIAL_ELO,
    )

    winner_match_count = match_counts.get(
        winner_id,
        0,
    )

    loser_match_count = match_counts.get(
        loser_id,
        0,
    )

    # Current match sonucu kullanılmadan önce kaydet.
    winner_surface_elo_before.append(winner_rating)
    loser_surface_elo_before.append(loser_rating)

    winner_surface_matches_before.append(
        winner_match_count
    )

    loser_surface_matches_before.append(
        loser_match_count
    )

    if match.match_status == "completed":
        (
            new_winner_rating,
            new_loser_rating,
            winner_elo_change,
            loser_elo_change,
        ) = update_elo(
            winner_rating=winner_rating,
            loser_rating=loser_rating,
            winner_matches_played=winner_match_count,
            loser_matches_played=loser_match_count,
        )

        ratings[winner_id] = new_winner_rating
        ratings[loser_id] = new_loser_rating

        match_counts[winner_id] = (
            winner_match_count + 1
        )

        match_counts[loser_id] = (
            loser_match_count + 1
        )

        postmatch_winner_surface_elo_change.append(
            winner_elo_change
        )

        postmatch_loser_surface_elo_change.append(
            loser_elo_change
        )

    else:
        postmatch_winner_surface_elo_change.append(
            np.nan
        )

        postmatch_loser_surface_elo_change.append(
            np.nan
        )

In [42]:
state_matches["winner_surface_elo_before"] = (
    winner_surface_elo_before
)

state_matches["loser_surface_elo_before"] = (
    loser_surface_elo_before
)

state_matches["winner_surface_matches_before"] = (
    winner_surface_matches_before
)

state_matches["loser_surface_matches_before"] = (
    loser_surface_matches_before
)

state_matches[
    "postmatch_winner_surface_elo_change"
] = postmatch_winner_surface_elo_change

state_matches[
    "postmatch_loser_surface_elo_change"
] = postmatch_loser_surface_elo_change

In [43]:
state_matches["p1_surface_elo"] = np.where(
    state_matches["winner_is_p1"],
    state_matches["winner_surface_elo_before"],
    state_matches["loser_surface_elo_before"],
)

state_matches["p2_surface_elo"] = np.where(
    state_matches["winner_is_p1"],
    state_matches["loser_surface_elo_before"],
    state_matches["winner_surface_elo_before"],
)

state_matches["p1_surface_matches_played"] = np.where(
    state_matches["winner_is_p1"],
    state_matches["winner_surface_matches_before"],
    state_matches["loser_surface_matches_before"],
)

state_matches["p2_surface_matches_played"] = np.where(
    state_matches["winner_is_p1"],
    state_matches["loser_surface_matches_before"],
    state_matches["winner_surface_matches_before"],
)

In [44]:
state_matches["surface_elo_difference"] = (
    state_matches["p1_surface_elo"]
    - state_matches["p2_surface_elo"]
)

state_matches["surface_elo_total"] = (
    state_matches["p1_surface_elo"]
    + state_matches["p2_surface_elo"]
)

state_matches["surface_experience_difference"] = (
    state_matches["p1_surface_matches_played"]
    - state_matches["p2_surface_matches_played"]
)

In [49]:
winner_surface_events = state_matches[
    [
        "surface",
        "winner_id",
        "winner_surface_elo_before",
        "winner_surface_matches_before",
    ]
].copy()

winner_surface_events["event_order"] = state_matches.index

winner_surface_events = winner_surface_events.rename(
    columns={
        "winner_id": "player_id",
        "winner_surface_elo_before": "elo_before",
        "winner_surface_matches_before": "matches_before",
    }
)


loser_surface_events = state_matches[
    [
        "surface",
        "loser_id",
        "loser_surface_elo_before",
        "loser_surface_matches_before",
    ]
].copy()

loser_surface_events["event_order"] = state_matches.index

loser_surface_events = loser_surface_events.rename(
    columns={
        "loser_id": "player_id",
        "loser_surface_elo_before": "elo_before",
        "loser_surface_matches_before": "matches_before",
    }
)


surface_events = (
    pd.concat(
        [winner_surface_events, loser_surface_events],
        ignore_index=True,
    )
    .dropna(subset=["elo_before"])
    .sort_values("event_order", kind="stable")
)

first_surface_events = (
    surface_events
    .groupby(
        ["surface", "player_id"],
        sort=False,
    )
    .head(1)
)


assert np.allclose(
    first_surface_events["elo_before"],
    INITIAL_ELO,
)

assert (
    first_surface_events["matches_before"] == 0
).all()

print("Surface Elo state engine correct")

Surface Elo state engine correct


In [52]:
player_name = "Roger Federer"

state_matches[
    (state_matches["p1_name"] == player_name)
    | (state_matches["p2_name"] == player_name)
][
    [
        "tourney_date",
        "tourney_name",
        "surface",
        "round",
        "p1_name",
        "p2_name",
        "p1_elo",
        "p2_elo",
        "p1_surface_elo",
        "p2_surface_elo",
        "target",
    ]
].tail(20)

,tourney_date,tourney_name,surface,round,p1_name,p2_name,p1_elo,p2_elo,p1_surface_elo,p2_surface_elo,target
119885,2020-01-20,Australian Open,Hard,R128,Steve Johnson,Roger Federer,1658.876765,2150.672057,1597.118205,2091.462416,0
119925,2020-01-20,Australian Open,Hard,R64,Filip Krajinovic,Roger Federer,1826.129207,2152.898927,1788.092316,2093.658631,0
119945,2020-01-20,Australian Open,Hard,R32,Roger Federer,John Millman,2158.189716,1796.880998,2099.535351,1755.011281,1
119955,2020-01-20,Australian Open,Hard,R16,Roger Federer,Marton Fucsovics,2162.632507,1746.051109,2104.374306,1752.520357,1
119960,2020-01-20,Australian Open,Hard,QF,Tennys Sandgren,Roger Federer,1719.124993,2165.965412,1669.524093,2109.036636,0
119963,2020-01-20,Australian Open,Hard,SF,Novak Djokovic,Roger Federer,2243.092247,2168.803325,2175.387117,2111.987799,1
121500,2021-03-08,Doha,Hard,R16,Roger Federer,Daniel Evans,2153.015738,1837.221555,2095.597382,1836.075056,1
121508,2021-03-08,Doha,Hard,QF,Nikoloz Basilashvili,Roger Federer,1581.304327,2158.603384,1586.455502,2102.930682,1
122106,2021-05-17,Geneva,Clay,R16,Roger Federer,Pablo Andujar,2119.994736,1543.688979,2030.211125,1577.645628,0
122210,2021-05-31,Roland Garros,Clay,R128,Roger Federer,Denis Istomin,2081.393787,1561.657664,1992.963366,1574.042721,1


In [53]:
player_names = {}

for match in state_matches.itertuples(index=False):
    player_names[int(match.winner_id)] = match.winner_name
    player_names[int(match.loser_id)] = match.loser_name


elo_leaderboard = pd.DataFrame(
    [
        {
            "player_id": player_id,
            "player_name": player_names.get(player_id),
            "elo": rating,
            "matches_played": general_match_counts[player_id],
        }
        for player_id, rating in general_ratings.items()
    ]
).sort_values(
    "elo",
    ascending=False,
).reset_index(drop=True)

elo_leaderboard.head(30)

,player_id,player_name,elo,matches_played
0,206173,Jannik Sinner,2350.741863,421
1,207989,Carlos Alcaraz,2238.021184,347
2,104417,Robin Soderling,2103.524465,454
3,100644,Alexander Zverev,2077.025742,754
4,103819,Roger Federer,2052.272689,1457
5,104925,Novak Djokovic,2038.499545,1313
6,209950,Arthur Fils,2025.742243,153
7,102158,Patrick Rafter,1996.682353,517
8,105223,Juan Martin del Potro,1980.799875,571
9,207733,Jack Draper,1957.638989,158


In [54]:
from collections import defaultdict


ROLLING_WINDOWS = (5, 10)


def summarize_elo_changes(history, window):
    recent_changes = history[-window:]

    elo_gained = sum(
        change
        for change in recent_changes
        if change > 0
    )

    elo_lost = sum(
        -change
        for change in recent_changes
        if change < 0
    )

    elo_net = elo_gained - elo_lost

    return {
        "gained": elo_gained,
        "lost": elo_lost,
        "net": elo_net,
    }

In [55]:
general_change_history = defaultdict(list)

surface_change_history = {
    surface: defaultdict(list)
    for surface in SUPPORTED_SURFACES
}

rolling_features = defaultdict(list)


for match in state_matches.itertuples(index=False):
    winner_id = int(match.winner_id)
    loser_id = int(match.loser_id)
    winner_is_p1 = bool(match.winner_is_p1)

    # General Elo: current maçtan önceki history
    winner_general_history = general_change_history[winner_id]
    loser_general_history = general_change_history[loser_id]

    for window in ROLLING_WINDOWS:
        winner_stats = summarize_elo_changes(
            winner_general_history,
            window,
        )

        loser_stats = summarize_elo_changes(
            loser_general_history,
            window,
        )

        if winner_is_p1:
            p1_stats = winner_stats
            p2_stats = loser_stats
        else:
            p1_stats = loser_stats
            p2_stats = winner_stats

        for metric in ("gained", "lost", "net"):
            rolling_features[
                f"p1_elo_{metric}_last_{window}"
            ].append(p1_stats[metric])

            rolling_features[
                f"p2_elo_{metric}_last_{window}"
            ].append(p2_stats[metric])

    # Surface Elo: yalnızca current surface geçmişi
    if match.surface in SUPPORTED_SURFACES:
        current_surface_history = surface_change_history[
            match.surface
        ]

        winner_surface_history = current_surface_history[
            winner_id
        ]

        loser_surface_history = current_surface_history[
            loser_id
        ]

        for window in ROLLING_WINDOWS:
            winner_stats = summarize_elo_changes(
                winner_surface_history,
                window,
            )

            loser_stats = summarize_elo_changes(
                loser_surface_history,
                window,
            )

            if winner_is_p1:
                p1_stats = winner_stats
                p2_stats = loser_stats
            else:
                p1_stats = loser_stats
                p2_stats = winner_stats

            for metric in ("gained", "lost", "net"):
                rolling_features[
                    f"p1_surface_elo_{metric}_last_{window}"
                ].append(p1_stats[metric])

                rolling_features[
                    f"p2_surface_elo_{metric}_last_{window}"
                ].append(p2_stats[metric])

    else:
        # Carpet model kapsamımızda değil.
        for window in ROLLING_WINDOWS:
            for player in ("p1", "p2"):
                for metric in ("gained", "lost", "net"):
                    rolling_features[
                        f"{player}_surface_elo_{metric}_last_{window}"
                    ].append(np.nan)

    # Current maç ancak feature'lar çıkarıldıktan sonra history'ye girer.
    if match.match_status == "completed":
        general_change_history[winner_id].append(
            float(match.postmatch_winner_elo_change)
        )

        general_change_history[loser_id].append(
            float(match.postmatch_loser_elo_change)
        )

        if match.surface in SUPPORTED_SURFACES:
            surface_change_history[match.surface][
                winner_id
            ].append(
                float(
                    match.postmatch_winner_surface_elo_change
                )
            )

            surface_change_history[match.surface][
                loser_id
            ].append(
                float(
                    match.postmatch_loser_surface_elo_change
                )
            )


rolling_feature_names = list(rolling_features)

state_matches[rolling_feature_names] = pd.DataFrame(
    rolling_features,
    index=state_matches.index,
)

print(
    f"{len(rolling_feature_names)} rolling Elo features created"
)

24 rolling Elo features created


In [56]:
# 1. Model satırlarında rolling feature eksik mi?
model_rows = state_matches[
    state_matches["is_model_eligible"]
]

assert not model_rows[
    rolling_feature_names
].isna().any().any()


# 2. gained, lost ve net matematiksel olarak tutarlı mı?
for scope in ("", "surface_"):
    for player in ("p1", "p2"):
        for window in ROLLING_WINDOWS:
            gained_column = (
                f"{player}_{scope}elo_gained_last_{window}"
            )

            lost_column = (
                f"{player}_{scope}elo_lost_last_{window}"
            )

            net_column = (
                f"{player}_{scope}elo_net_last_{window}"
            )

            assert (
                model_rows[gained_column] >= 0
            ).all()

            assert (
                model_rows[lost_column] >= 0
            ).all()

            assert np.allclose(
                model_rows[net_column],
                (
                    model_rows[gained_column]
                    - model_rows[lost_column]
                ),
            )


# 3. Son 10 maç toplamı, son 5 maç toplamından küçük olamaz.
for scope in ("", "surface_"):
    for player in ("p1", "p2"):
        for metric in ("gained", "lost"):
            last_5 = (
                f"{player}_{scope}elo_{metric}_last_5"
            )

            last_10 = (
                f"{player}_{scope}elo_{metric}_last_10"
            )

            assert (
                model_rows[last_10]
                >= model_rows[last_5]
            ).all()


print("Basic rolling Elo checks passed")

Basic rolling Elo checks passed


In [57]:
completed_matches = state_matches[
    state_matches["match_status"].eq("completed")
].copy()


winner_events = pd.DataFrame(
    {
        "event_order": completed_matches.index,
        "player_id": completed_matches["winner_id"],
        "elo_change": completed_matches[
            "postmatch_winner_elo_change"
        ],
    }
)

loser_events = pd.DataFrame(
    {
        "event_order": completed_matches.index,
        "player_id": completed_matches["loser_id"],
        "elo_change": completed_matches[
            "postmatch_loser_elo_change"
        ],
    }
)


for window in ROLLING_WINDOWS:
    winner_events[f"actual_net_{window}"] = np.where(
        completed_matches["winner_is_p1"],
        completed_matches[
            f"p1_elo_net_last_{window}"
        ],
        completed_matches[
            f"p2_elo_net_last_{window}"
        ],
    )

    loser_events[f"actual_net_{window}"] = np.where(
        completed_matches["winner_is_p1"],
        completed_matches[
            f"p2_elo_net_last_{window}"
        ],
        completed_matches[
            f"p1_elo_net_last_{window}"
        ],
    )


general_events = (
    pd.concat(
        [winner_events, loser_events],
        ignore_index=True,
    )
    .sort_values("event_order", kind="stable")
)


for window in ROLLING_WINDOWS:
    general_events[f"expected_net_{window}"] = (
        general_events
        .groupby("player_id", sort=False)["elo_change"]
        .transform(
            lambda changes: (
                changes
                .shift(1)
                .rolling(
                    window,
                    min_periods=1,
                )
                .sum()
                .fillna(0)
            )
        )
    )

    assert np.allclose(
        general_events[f"actual_net_{window}"],
        general_events[f"expected_net_{window}"],
    )


print("Rolling Elo is chronological and leakage-free")

Rolling Elo is chronological and leakage-free


In [58]:
player_name = "Novak Djokovic"

player_matches = state_matches[
    (
        state_matches["winner_name"].eq(player_name)
        | state_matches["loser_name"].eq(player_name)
    )
    & state_matches["match_status"].eq("completed")
].copy()


player_is_p1 = player_matches["p1_name"].eq(player_name)
player_won = player_matches["winner_name"].eq(player_name)


player_matches["opponent"] = np.where(
    player_won,
    player_matches["loser_name"],
    player_matches["winner_name"],
)

player_matches["result"] = np.where(
    player_won,
    "W",
    "L",
)

player_matches["player_elo_before"] = np.where(
    player_won,
    player_matches["winner_elo_before"],
    player_matches["loser_elo_before"],
)

player_matches["postmatch_elo_change"] = np.where(
    player_won,
    player_matches["postmatch_winner_elo_change"],
    player_matches["postmatch_loser_elo_change"],
)


for window in ROLLING_WINDOWS:
    for metric in ("gained", "lost", "net"):
        player_matches[
            f"elo_{metric}_last_{window}"
        ] = np.where(
            player_is_p1,
            player_matches[
                f"p1_elo_{metric}_last_{window}"
            ],
            player_matches[
                f"p2_elo_{metric}_last_{window}"
            ],
        )


player_matches[
    [
        "tourney_date",
        "tourney_name",
        "surface",
        "round",
        "opponent",
        "result",
        "player_elo_before",
        "postmatch_elo_change",
        "elo_gained_last_5",
        "elo_lost_last_5",
        "elo_net_last_5",
        "elo_net_last_10",
    ]
].tail(15)

,tourney_date,tourney_name,surface,round,opponent,result,player_elo_before,postmatch_elo_change,elo_gained_last_5,elo_lost_last_5,elo_net_last_5,elo_net_last_10
134513,2025-11-02,Athens,Hard,QF,Nuno Borges,W,2082.222466,4.254982,16.841231,36.937661,-20.096431,-4.420962
134515,2025-11-02,Athens,Hard,SF,Yannick Hanfmann,W,2086.477447,4.425677,17.875575,36.937661,-19.062087,-5.686035
134516,2025-11-02,Athens,Hard,F,Lorenzo Musetti,W,2090.903124,10.178901,16.895390,36.937661,-20.042271,-5.638611
134731,2026-01-19,Australian Open,Hard,R128,Pedro Martinez,W,2101.082025,1.259193,23.668158,36.937661,-13.269503,-8.874233
134771,2026-01-19,Australian Open,Hard,R64,Francesco Maestrelli,W,2102.341218,1.381622,24.927351,0.000000,24.927351,4.294922
134791,2026-01-19,Australian Open,Hard,R32,Botic Van De Zandschulp,W,2103.722840,3.832942,21.500374,0.000000,21.500374,1.403943
134809,2026-01-19,Australian Open,Hard,SF,Jannik Sinner,W,2107.555782,32.783430,21.078334,0.000000,21.078334,2.016247
134810,2026-01-19,Australian Open,Hard,F,Carlos Alcaraz,L,2140.339211,-12.741755,49.436087,0.000000,49.436087,29.393816
135160,2026-03-04,Indian Wells Masters,Hard,R64,Kamil Majchrzak,W,2127.597456,3.080457,39.257186,12.741755,26.515431,13.245928
135180,2026-03-04,Indian Wells Masters,Hard,R32,Aleksandar Kovacevic,W,2130.677913,1.606579,41.078450,12.741755,28.336695,53.264046


In [59]:
example_position = len(player_matches) - 1

previous_5_matches = player_matches.iloc[
    example_position - 5:example_position
]

current_match = player_matches.iloc[
    example_position
]

display(
    previous_5_matches[
        [
            "tourney_date",
            "opponent",
            "result",
            "postmatch_elo_change",
        ]
    ]
)

print(
    "Previous 5 change total:",
    previous_5_matches[
        "postmatch_elo_change"
    ].sum(),
)

print(
    "Current row elo_net_last_5:",
    current_match["elo_net_last_5"],
)

,tourney_date,opponent,result,postmatch_elo_change
135180,2026-03-04,Aleksandar Kovacevic,W,1.606579
135190,2026-03-04,Jack Draper,L,-27.963233
135628,2026-05-06,Dino Prizmic,L,-37.682016
135788,2026-05-25,Giovanni Mpetshi Perricard,W,2.612630
135828,2026-05-25,Valentin Royer,W,1.710290


Previous 5 change total: -59.715751384064106
Current row elo_net_last_5: -59.715751384064106


In [60]:
h2h_wins = defaultdict(
    lambda: defaultdict(int)
)

surface_h2h_wins = {
    surface: defaultdict(
        lambda: defaultdict(int)
    )
    for surface in SUPPORTED_SURFACES
}


h2h_features = defaultdict(list)


for match in state_matches.itertuples(index=False):
    p1_id = int(match.p1_id)
    p2_id = int(match.p2_id)

    pair_key = tuple(
        sorted((p1_id, p2_id))
    )

    # General H2H: current maçtan önceki durum
    pair_history = h2h_wins[pair_key]

    p1_wins = pair_history[p1_id]
    p2_wins = pair_history[p2_id]

    total_matches = p1_wins + p2_wins

    p1_win_rate = (
        p1_wins / total_matches
        if total_matches > 0
        else 0.5
    )

    h2h_features[
        "h2h_matches_before"
    ].append(total_matches)

    h2h_features[
        "p1_h2h_wins_before"
    ].append(p1_wins)

    h2h_features[
        "p2_h2h_wins_before"
    ].append(p2_wins)

    h2h_features[
        "p1_h2h_win_rate_before"
    ].append(p1_win_rate)

    h2h_features[
        "h2h_win_difference"
    ].append(p1_wins - p2_wins)

    # Surface-specific H2H
    if match.surface in SUPPORTED_SURFACES:
        surface_pair_history = surface_h2h_wins[
            match.surface
        ][pair_key]

        p1_surface_wins = surface_pair_history[
            p1_id
        ]

        p2_surface_wins = surface_pair_history[
            p2_id
        ]

        total_surface_matches = (
            p1_surface_wins
            + p2_surface_wins
        )

        p1_surface_win_rate = (
            p1_surface_wins / total_surface_matches
            if total_surface_matches > 0
            else 0.5
        )

        h2h_features[
            "surface_h2h_matches_before"
        ].append(total_surface_matches)

        h2h_features[
            "p1_surface_h2h_wins_before"
        ].append(p1_surface_wins)

        h2h_features[
            "p2_surface_h2h_wins_before"
        ].append(p2_surface_wins)

        h2h_features[
            "p1_surface_h2h_win_rate_before"
        ].append(p1_surface_win_rate)

        h2h_features[
            "surface_h2h_win_difference"
        ].append(
            p1_surface_wins
            - p2_surface_wins
        )

    else:
        h2h_features[
            "surface_h2h_matches_before"
        ].append(np.nan)

        h2h_features[
            "p1_surface_h2h_wins_before"
        ].append(np.nan)

        h2h_features[
            "p2_surface_h2h_wins_before"
        ].append(np.nan)

        h2h_features[
            "p1_surface_h2h_win_rate_before"
        ].append(np.nan)

        h2h_features[
            "surface_h2h_win_difference"
        ].append(np.nan)

    # Current sonuç feature çıkarıldıktan sonra eklenir.
    if match.match_status == "completed":
        winner_id = int(match.winner_id)

        h2h_wins[pair_key][winner_id] += 1

        if match.surface in SUPPORTED_SURFACES:
            surface_h2h_wins[
                match.surface
            ][pair_key][winner_id] += 1


h2h_feature_names = list(h2h_features)

state_matches[h2h_feature_names] = pd.DataFrame(
    h2h_features,
    index=state_matches.index,
)

print(
    f"{len(h2h_feature_names)} H2H features created"
)

10 H2H features created


In [61]:
model_rows = state_matches[
    state_matches["is_model_eligible"]
].copy()


# Model kapsamındaki satırlarda eksik H2H feature olmamalı.
assert not model_rows[
    h2h_feature_names
].isna().any().any()


# General H2H toplamı doğru mu?
assert (
    model_rows["h2h_matches_before"]
    == (
        model_rows["p1_h2h_wins_before"]
        + model_rows["p2_h2h_wins_before"]
    )
).all()


# General H2H difference doğru mu?
assert (
    model_rows["h2h_win_difference"]
    == (
        model_rows["p1_h2h_wins_before"]
        - model_rows["p2_h2h_wins_before"]
    )
).all()


# Surface H2H toplamı doğru mu?
assert (
    model_rows["surface_h2h_matches_before"]
    == (
        model_rows["p1_surface_h2h_wins_before"]
        + model_rows["p2_surface_h2h_wins_before"]
    )
).all()


# Surface H2H difference doğru mu?
assert (
    model_rows["surface_h2h_win_difference"]
    == (
        model_rows["p1_surface_h2h_wins_before"]
        - model_rows["p2_surface_h2h_wins_before"]
    )
).all()


# Win rate değerleri doğru mu?
expected_general_rate = np.where(
    model_rows["h2h_matches_before"] > 0,
    (
        model_rows["p1_h2h_wins_before"]
        / model_rows["h2h_matches_before"]
    ),
    0.5,
)

expected_surface_rate = np.where(
    model_rows["surface_h2h_matches_before"] > 0,
    (
        model_rows["p1_surface_h2h_wins_before"]
        / model_rows["surface_h2h_matches_before"]
    ),
    0.5,
)

assert np.allclose(
    model_rows["p1_h2h_win_rate_before"],
    expected_general_rate,
)

assert np.allclose(
    model_rows["p1_surface_h2h_win_rate_before"],
    expected_surface_rate,
)


print("H2H consistency checks passed")

H2H consistency checks passed


In [62]:
h2h_validation = state_matches[
    [
        "p1_id",
        "p2_id",
        "surface",
        "match_status",
        "h2h_matches_before",
        "surface_h2h_matches_before",
    ]
].copy()


h2h_validation["pair_key"] = [
    tuple(sorted((int(p1_id), int(p2_id))))
    for p1_id, p2_id in zip(
        h2h_validation["p1_id"],
        h2h_validation["p2_id"],
    )
]

h2h_validation["completed"] = (
    h2h_validation["match_status"]
    .eq("completed")
    .astype(int)
)


# Current completed flag çıkarılınca yalnızca önceki maçlar kalır.
h2h_validation["expected_h2h_matches_before"] = (
    h2h_validation
    .groupby("pair_key", sort=False)["completed"]
    .cumsum()
    - h2h_validation["completed"]
)

assert (
    h2h_validation["h2h_matches_before"]
    == h2h_validation[
        "expected_h2h_matches_before"
    ]
).all()


supported_surface_rows = h2h_validation[
    h2h_validation["surface"].isin(
        SUPPORTED_SURFACES
    )
].copy()

supported_surface_rows[
    "expected_surface_h2h_matches_before"
] = (
    supported_surface_rows
    .groupby(
        ["surface", "pair_key"],
        sort=False,
    )["completed"]
    .cumsum()
    - supported_surface_rows["completed"]
)

assert (
    supported_surface_rows[
        "surface_h2h_matches_before"
    ]
    == supported_surface_rows[
        "expected_surface_h2h_matches_before"
    ]
).all()


print("H2H is chronological and leakage-free")

H2H is chronological and leakage-free


In [63]:
player_a = "Rafael Nadal"
player_b = "Roger Federer"

state_matches[
    (
        state_matches["p1_name"].isin(
            [player_a, player_b]
        )
    )
    & (
        state_matches["p2_name"].isin(
            [player_a, player_b]
        )
    )
][
    [
        "tourney_date",
        "tourney_name",
        "surface",
        "round",
        "p1_name",
        "p2_name",
        "p1_h2h_wins_before",
        "p2_h2h_wins_before",
        "h2h_matches_before",
        "p1_h2h_win_rate_before",
        "surface_h2h_matches_before",
        "target",
    ]
]

,tourney_date,tourney_name,surface,round,p1_name,p2_name,p1_h2h_wins_before,p2_h2h_wins_before,h2h_matches_before,p1_h2h_win_rate_before,surface_h2h_matches_before,target
76591,2004-03-22,Miami Masters,Hard,R32,Rafael Nadal,Roger Federer,0,0,0,0.500000,0.0,1
79619,2005-03-21,Miami Masters,Hard,F,Rafael Nadal,Roger Federer,1,0,1,1.000000,1.0,0
80200,2005-05-23,Roland Garros,Clay,SF,Roger Federer,Rafael Nadal,1,1,2,0.500000,0.0,0
82333,2006-02-27,Dubai,Hard,F,Roger Federer,Rafael Nadal,1,2,3,0.333333,2.0,0
82679,2006-04-17,Monte Carlo Masters,Clay,F,Rafael Nadal,Roger Federer,3,1,4,0.750000,1.0,1
82890,2006-05-08,Rome Masters,Clay,F,Rafael Nadal,Roger Federer,4,1,5,0.800000,2.0,1
83137,2006-05-29,Roland Garros,Clay,F,Roger Federer,Rafael Nadal,1,5,6,0.166667,3.0,0
83412,2006-06-26,Wimbledon,Grass,F,Roger Federer,Rafael Nadal,1,6,7,0.142857,0.0,1
84643,2006-11-13,Masters Cup,Hard,SF,Rafael Nadal,Roger Federer,6,2,8,0.750000,3.0,0
85638,2007-04-15,Monte Carlo Masters,Clay,F,Roger Federer,Rafael Nadal,3,6,9,0.333333,4.0,0


In [66]:
def create_tournament_state():
    return {
        "matches": 0,
        "wins": 0,
        "losses": 0,
        "elo_gained": 0.0,
        "elo_lost": 0.0,
    }


tournament_history = defaultdict(
    create_tournament_state
)

tournament_features = defaultdict(list)


for match in state_matches.itertuples(index=False):
    tourney_id = match.tourney_id

    p1_id = int(match.p1_id)
    p2_id = int(match.p2_id)

    p1_key = (tourney_id, p1_id)
    p2_key = (tourney_id, p2_id)

    # Current maçtan önceki tournament history
    p1_history = tournament_history[p1_key]
    p2_history = tournament_history[p2_key]

    for player, history in (
        ("p1", p1_history),
        ("p2", p2_history),
    ):
        matches_before = history["matches"]

        win_rate_before = (
            history["wins"] / matches_before
            if matches_before > 0
            else 0.5
        )

        tournament_features[
            f"{player}_tourney_matches_before"
        ].append(matches_before)

        tournament_features[
            f"{player}_tourney_wins_before"
        ].append(history["wins"])

        tournament_features[
            f"{player}_tourney_losses_before"
        ].append(history["losses"])

        tournament_features[
            f"{player}_tourney_win_rate_before"
        ].append(win_rate_before)

        tournament_features[
            f"{player}_tourney_elo_gained_before"
        ].append(history["elo_gained"])

        tournament_features[
            f"{player}_tourney_elo_lost_before"
        ].append(history["elo_lost"])

        tournament_features[
            f"{player}_tourney_elo_net_before"
        ].append(
            history["elo_gained"]
            - history["elo_lost"]
        )

    # Current maç yalnızca feature çıkarıldıktan sonra eklenir.
    if match.match_status == "completed":
        winner_id = int(match.winner_id)
        loser_id = int(match.loser_id)

        winner_key = (tourney_id, winner_id)
        loser_key = (tourney_id, loser_id)

        winner_history = tournament_history[
            winner_key
        ]

        loser_history = tournament_history[
            loser_key
        ]

        winner_history["matches"] += 1
        winner_history["wins"] += 1
        winner_history["elo_gained"] += float(
            match.postmatch_winner_elo_change
        )

        loser_history["matches"] += 1
        loser_history["losses"] += 1
        loser_history["elo_lost"] += -float(
            match.postmatch_loser_elo_change
        )


tournament_feature_names = list(
    tournament_features
)

tournament_feature_frame = pd.DataFrame(
    tournament_features,
    index=state_matches.index,
)

state_matches = pd.concat(
    [
        state_matches.drop(
            columns=tournament_feature_names,
            errors="ignore",
        ),
        tournament_feature_frame,
    ],
    axis=1,
).copy()

print(
    f"{len(tournament_feature_names)} "
    "player-level tournament features created"
)

14 player-level tournament features created


In [67]:
tournament_comparison_metrics = (
    "matches",
    "wins",
    "losses",
    "win_rate",
    "elo_gained",
    "elo_lost",
    "elo_net",
)


tournament_comparison_features = {
    f"tourney_{metric}_difference": (
        state_matches[
            f"p1_tourney_{metric}_before"
        ]
        - state_matches[
            f"p2_tourney_{metric}_before"
        ]
    )
    for metric in tournament_comparison_metrics
}

tournament_comparison_feature_names = list(
    tournament_comparison_features
)

state_matches = pd.concat(
    [
        state_matches.drop(
            columns=tournament_comparison_feature_names,
            errors="ignore",
        ),
        pd.DataFrame(
            tournament_comparison_features,
            index=state_matches.index,
        ),
    ],
    axis=1,
).copy()


print(
    f"{len(tournament_comparison_metrics)} "
    "tournament comparison features created"
)

7 tournament comparison features created


In [68]:
model_rows = state_matches[
    state_matches["is_model_eligible"]
].copy()


# Model satırlarında eksik değer olmamalı.
all_tournament_feature_names = (
    tournament_feature_names
    + tournament_comparison_feature_names
)

assert not model_rows[
    all_tournament_feature_names
].isna().any().any()


# Matches = wins + losses olmalı.
for player in ("p1", "p2"):
    assert (
        model_rows[
            f"{player}_tourney_matches_before"
        ]
        == (
            model_rows[
                f"{player}_tourney_wins_before"
            ]
            + model_rows[
                f"{player}_tourney_losses_before"
            ]
        )
    ).all()


# Net Elo = gained - lost olmalı.
for player in ("p1", "p2"):
    assert np.allclose(
        model_rows[
            f"{player}_tourney_elo_net_before"
        ],
        (
            model_rows[
                f"{player}_tourney_elo_gained_before"
            ]
            - model_rows[
                f"{player}_tourney_elo_lost_before"
            ]
        ),
    )


# Win rate doğru hesaplanmış mı?
for player in ("p1", "p2"):
    matches = model_rows[
        f"{player}_tourney_matches_before"
    ]

    wins = model_rows[
        f"{player}_tourney_wins_before"
    ]

    expected_win_rate = np.where(
        matches > 0,
        wins / matches,
        0.5,
    )

    assert np.allclose(
        model_rows[
            f"{player}_tourney_win_rate_before"
        ],
        expected_win_rate,
    )


# P1-P2 difference kolonları doğru mu?
for metric in tournament_comparison_metrics:
    assert np.allclose(
        model_rows[
            f"tourney_{metric}_difference"
        ],
        (
            model_rows[
                f"p1_tourney_{metric}_before"
            ]
            - model_rows[
                f"p2_tourney_{metric}_before"
            ]
        ),
    )


print("Tournament feature consistency checks passed")

Tournament feature consistency checks passed


In [69]:
p1_tournament_events = pd.DataFrame(
    {
        "event_order": state_matches.index,
        "tourney_id": state_matches["tourney_id"],
        "player_id": state_matches["p1_id"],
        "completed": (
            state_matches["match_status"]
            .eq("completed")
            .astype(int)
        ),
        "actual_matches_before": state_matches[
            "p1_tourney_matches_before"
        ],
    }
)

p2_tournament_events = pd.DataFrame(
    {
        "event_order": state_matches.index,
        "tourney_id": state_matches["tourney_id"],
        "player_id": state_matches["p2_id"],
        "completed": (
            state_matches["match_status"]
            .eq("completed")
            .astype(int)
        ),
        "actual_matches_before": state_matches[
            "p2_tourney_matches_before"
        ],
    }
)


tournament_events = (
    pd.concat(
        [
            p1_tournament_events,
            p2_tournament_events,
        ],
        ignore_index=True,
    )
    .sort_values(
        "event_order",
        kind="stable",
    )
)


tournament_events[
    "expected_matches_before"
] = (
    tournament_events
    .groupby(
        ["tourney_id", "player_id"],
        sort=False,
    )["completed"]
    .cumsum()
    - tournament_events["completed"]
)


assert (
    tournament_events["actual_matches_before"]
    == tournament_events["expected_matches_before"]
).all()


print(
    "Tournament features are chronological "
    "and leakage-free"
)


Tournament features are chronological and leakage-free


In [70]:
import re


def parse_tennis_score(score):
    result = {
        "postmatch_winner_sets_won": np.nan,
        "postmatch_loser_sets_won": np.nan,
        "postmatch_winner_games_won": np.nan,
        "postmatch_loser_games_won": np.nan,
        "score_parse_success": False,
    }

    if pd.isna(score):
        return result

    winner_sets = 0
    loser_sets = 0

    winner_games = 0
    loser_games = 0

    parsed_sets = 0

    for token in str(score).split():
        upper_token = token.upper()

        if upper_token in {
            "RET",
            "W/O",
            "DEF",
            "ABD",
            "UNK",
        }:
            continue

        # [10-8] gibi match tiebreak'leri set olarak sayacağız,
        # fakat point sayılarını normal game toplamına katmayacağız.
        is_match_tiebreak = (
            token.startswith("[")
            and token.endswith("]")
        )

        cleaned_token = token.strip("[]")

        match_result = re.fullmatch(
            r"(\d+)-(\d+)(?:\(\d+\))?",
            cleaned_token,
        )

        if match_result is None:
            continue

        winner_score = int(
            match_result.group(1)
        )

        loser_score = int(
            match_result.group(2)
        )

        if winner_score == loser_score:
            continue

        parsed_sets += 1

        if winner_score > loser_score:
            winner_sets += 1
        else:
            loser_sets += 1

        if not is_match_tiebreak:
            winner_games += winner_score
            loser_games += loser_score

    if parsed_sets == 0:
        return result

    result.update(
        {
            "postmatch_winner_sets_won": winner_sets,
            "postmatch_loser_sets_won": loser_sets,
            "postmatch_winner_games_won": winner_games,
            "postmatch_loser_games_won": loser_games,
            "score_parse_success": True,
        }
    )

    return result

In [71]:
score_feature_frame = pd.DataFrame(
    state_matches["score"]
    .apply(parse_tennis_score)
    .tolist(),
    index=state_matches.index,
)


state_matches = pd.concat(
    [
        state_matches.drop(
            columns=score_feature_frame.columns,
            errors="ignore",
        ),
        score_feature_frame,
    ],
    axis=1,
).copy()


completed_score_parse_rate = (
    state_matches.loc[
        state_matches["match_status"].eq("completed"),
        "score_parse_success",
    ]
    .mean()
)


print(
    "Completed score parse rate:",
    f"{completed_score_parse_rate:.2%}",
)

Completed score parse rate: 100.00%


In [72]:
def create_tournament_score_state():
    return {
        "sets_won": 0,
        "sets_lost": 0,
        "games_won": 0,
        "games_lost": 0,
    }


tournament_score_history = defaultdict(
    create_tournament_score_state
)

tournament_score_features = defaultdict(list)


for match in state_matches.itertuples(index=False):
    tourney_id = match.tourney_id

    p1_id = int(match.p1_id)
    p2_id = int(match.p2_id)

    p1_key = (tourney_id, p1_id)
    p2_key = (tourney_id, p2_id)

    p1_history = tournament_score_history[p1_key]
    p2_history = tournament_score_history[p2_key]

    # Önceki round'lardan feature çıkar.
    for player, history in (
        ("p1", p1_history),
        ("p2", p2_history),
    ):
        sets_played = (
            history["sets_won"]
            + history["sets_lost"]
        )

        games_played = (
            history["games_won"]
            + history["games_lost"]
        )

        set_win_rate = (
            history["sets_won"] / sets_played
            if sets_played > 0
            else 0.5
        )

        game_win_rate = (
            history["games_won"] / games_played
            if games_played > 0
            else 0.5
        )

        values = {
            "sets_won": history["sets_won"],
            "sets_lost": history["sets_lost"],
            "sets_net": (
                history["sets_won"]
                - history["sets_lost"]
            ),
            "set_win_rate": set_win_rate,
            "games_won": history["games_won"],
            "games_lost": history["games_lost"],
            "games_net": (
                history["games_won"]
                - history["games_lost"]
            ),
            "game_win_rate": game_win_rate,
        }

        for metric, value in values.items():
            tournament_score_features[
                f"{player}_tourney_{metric}_before"
            ].append(value)

    # Current score yalnızca feature çıkarıldıktan sonra eklenir.
    if (
        match.match_status == "completed"
        and match.score_parse_success
    ):
        winner_key = (
            tourney_id,
            int(match.winner_id),
        )

        loser_key = (
            tourney_id,
            int(match.loser_id),
        )

        winner_history = tournament_score_history[
            winner_key
        ]

        loser_history = tournament_score_history[
            loser_key
        ]

        winner_sets = int(
            match.postmatch_winner_sets_won
        )

        loser_sets = int(
            match.postmatch_loser_sets_won
        )

        winner_games = int(
            match.postmatch_winner_games_won
        )

        loser_games = int(
            match.postmatch_loser_games_won
        )

        # Winner açısından
        winner_history["sets_won"] += winner_sets
        winner_history["sets_lost"] += loser_sets
        winner_history["games_won"] += winner_games
        winner_history["games_lost"] += loser_games

        # Loser açısından perspektif tersine çevrilir.
        loser_history["sets_won"] += loser_sets
        loser_history["sets_lost"] += winner_sets
        loser_history["games_won"] += loser_games
        loser_history["games_lost"] += winner_games


tournament_score_feature_names = list(
    tournament_score_features
)

tournament_score_feature_frame = pd.DataFrame(
    tournament_score_features,
    index=state_matches.index,
)


state_matches = pd.concat(
    [
        state_matches.drop(
            columns=tournament_score_feature_names,
            errors="ignore",
        ),
        tournament_score_feature_frame,
    ],
    axis=1,
).copy()


print(
    f"{len(tournament_score_feature_names)} "
    "player-level tournament score features created"
)

16 player-level tournament score features created


In [73]:
tournament_score_metrics = (
    "sets_won",
    "sets_lost",
    "sets_net",
    "set_win_rate",
    "games_won",
    "games_lost",
    "games_net",
    "game_win_rate",
)


tournament_score_comparisons = {
    f"tourney_{metric}_difference": (
        state_matches[
            f"p1_tourney_{metric}_before"
        ]
        - state_matches[
            f"p2_tourney_{metric}_before"
        ]
    )
    for metric in tournament_score_metrics
}

tournament_score_comparison_names = list(
    tournament_score_comparisons
)


state_matches = pd.concat(
    [
        state_matches.drop(
            columns=tournament_score_comparison_names,
            errors="ignore",
        ),
        pd.DataFrame(
            tournament_score_comparisons,
            index=state_matches.index,
        ),
    ],
    axis=1,
).copy()


print(
    f"{len(tournament_score_comparison_names)} "
    "tournament score comparison features created"
)

8 tournament score comparison features created


In [74]:
model_rows = state_matches[
    state_matches["is_model_eligible"]
].copy()


all_tournament_score_features = (
    tournament_score_feature_names
    + tournament_score_comparison_names
)

assert not model_rows[
    all_tournament_score_features
].isna().any().any()


for player in ("p1", "p2"):
    # Net değerleri
    assert np.allclose(
        model_rows[
            f"{player}_tourney_sets_net_before"
        ],
        (
            model_rows[
                f"{player}_tourney_sets_won_before"
            ]
            - model_rows[
                f"{player}_tourney_sets_lost_before"
            ]
        ),
    )

    assert np.allclose(
        model_rows[
            f"{player}_tourney_games_net_before"
        ],
        (
            model_rows[
                f"{player}_tourney_games_won_before"
            ]
            - model_rows[
                f"{player}_tourney_games_lost_before"
            ]
        ),
    )

    # Rate değerleri 0–1 arasında olmalı.
    assert model_rows[
        f"{player}_tourney_set_win_rate_before"
    ].between(0, 1).all()

    assert model_rows[
        f"{player}_tourney_game_win_rate_before"
    ].between(0, 1).all()


# Comparison kolonları doğru mu?
for metric in tournament_score_metrics:
    assert np.allclose(
        model_rows[
            f"tourney_{metric}_difference"
        ],
        (
            model_rows[
                f"p1_tourney_{metric}_before"
            ]
            - model_rows[
                f"p2_tourney_{metric}_before"
            ]
        ),
    )


print("Tournament score consistency checks passed")

Tournament score consistency checks passed


In [75]:
p1_first_event_columns = {
    "p1_id": "player_id",
    "p1_tourney_sets_won_before": "sets_won",
    "p1_tourney_sets_lost_before": "sets_lost",
    "p1_tourney_games_won_before": "games_won",
    "p1_tourney_games_lost_before": "games_lost",
}

p2_first_event_columns = {
    "p2_id": "player_id",
    "p2_tourney_sets_won_before": "sets_won",
    "p2_tourney_sets_lost_before": "sets_lost",
    "p2_tourney_games_won_before": "games_won",
    "p2_tourney_games_lost_before": "games_lost",
}


p1_tournament_score_events = state_matches[
    [
        "tourney_id",
        *p1_first_event_columns,
    ]
].rename(
    columns=p1_first_event_columns
)

p1_tournament_score_events["event_order"] = (
    state_matches.index
)


p2_tournament_score_events = state_matches[
    [
        "tourney_id",
        *p2_first_event_columns,
    ]
].rename(
    columns=p2_first_event_columns
)

p2_tournament_score_events["event_order"] = (
    state_matches.index
)


first_tournament_score_events = (
    pd.concat(
        [
            p1_tournament_score_events,
            p2_tournament_score_events,
        ],
        ignore_index=True,
    )
    .sort_values(
        "event_order",
        kind="stable",
    )
    .groupby(
        ["tourney_id", "player_id"],
        sort=False,
    )
    .head(1)
)


initial_score_columns = [
    "sets_won",
    "sets_lost",
    "games_won",
    "games_lost",
]

assert (
    first_tournament_score_events[
        initial_score_columns
    ] == 0
).all().all()


print(
    "Tournament score features are "
    "chronological and leakage-free"
)


Tournament score features are chronological and leakage-free


In [76]:
player_a = "Novak Djokovic"
player_b = "Carlos Alcaraz"
tournament_name = "Wimbledon"


example_candidates = state_matches[
    state_matches["tourney_name"].str.contains(
        tournament_name,
        case=False,
        na=False,
    )
    & state_matches["p1_name"].isin(
        [player_a, player_b]
    )
    & state_matches["p2_name"].isin(
        [player_a, player_b]
    )
].sort_values(
    ["tourney_date", "round_order"]
)


example_match = example_candidates.iloc[-1]

example_match[
    [
        "tourney_date",
        "tourney_name",
        "surface",
        "round",
        "p1_name",
        "p2_name",
        "p1_tourney_matches_before",
        "p2_tourney_matches_before",
        "p1_tourney_elo_net_before",
        "p2_tourney_elo_net_before",
        "p1_tourney_sets_won_before",
        "p1_tourney_sets_lost_before",
        "p2_tourney_sets_won_before",
        "p2_tourney_sets_lost_before",
        "p1_tourney_games_won_before",
        "p1_tourney_games_lost_before",
        "p2_tourney_games_won_before",
        "p2_tourney_games_lost_before",
        "p1_tourney_game_win_rate_before",
        "p2_tourney_game_win_rate_before",
        "score",
        "target",
    ]
]

tourney_date                       2024-07-01 00:00:00
tourney_name                                 Wimbledon
surface                                          Grass
round                                                F
p1_name                                 Carlos Alcaraz
p2_name                                 Novak Djokovic
p1_tourney_matches_before                            6
p2_tourney_matches_before                            5
p1_tourney_elo_net_before                    38.032709
p2_tourney_elo_net_before                    17.799297
p1_tourney_sets_won_before                          18
p1_tourney_sets_lost_before                          5
p2_tourney_sets_won_before                          15
p2_tourney_sets_lost_before                          2
p1_tourney_games_won_before                        134
p1_tourney_games_lost_before                        97
p2_tourney_games_won_before                        102
p2_tourney_games_lost_before                        66
p1_tourney

In [77]:
example_index = example_match.name
example_tourney_id = example_match["tourney_id"]


for player_name in (
    example_match["p1_name"],
    example_match["p2_name"],
):
    previous_matches = state_matches[
        state_matches["tourney_id"].eq(
            example_tourney_id
        )
        & (state_matches.index < example_index)
        & state_matches["match_status"].eq(
            "completed"
        )
        & (
            state_matches["winner_name"].eq(
                player_name
            )
            | state_matches["loser_name"].eq(
                player_name
            )
        )
    ].copy()

    player_won = previous_matches[
        "winner_name"
    ].eq(player_name)

    previous_matches["player_sets_won"] = np.where(
        player_won,
        previous_matches[
            "postmatch_winner_sets_won"
        ],
        previous_matches[
            "postmatch_loser_sets_won"
        ],
    )

    previous_matches["player_sets_lost"] = np.where(
        player_won,
        previous_matches[
            "postmatch_loser_sets_won"
        ],
        previous_matches[
            "postmatch_winner_sets_won"
        ],
    )

    previous_matches["player_games_won"] = np.where(
        player_won,
        previous_matches[
            "postmatch_winner_games_won"
        ],
        previous_matches[
            "postmatch_loser_games_won"
        ],
    )

    previous_matches["player_games_lost"] = np.where(
        player_won,
        previous_matches[
            "postmatch_loser_games_won"
        ],
        previous_matches[
            "postmatch_winner_games_won"
        ],
    )

    print(f"\n{player_name} — previous rounds")

    display(
        previous_matches[
            [
                "round",
                "winner_name",
                "loser_name",
                "score",
                "player_sets_won",
                "player_sets_lost",
                "player_games_won",
                "player_games_lost",
            ]
        ]
    )

    print(
        "Totals:",
        previous_matches[
            [
                "player_sets_won",
                "player_sets_lost",
                "player_games_won",
                "player_games_lost",
            ]
        ].sum().to_dict(),
    )


Carlos Alcaraz — previous rounds


,round,winner_name,loser_name,score,player_sets_won,player_sets_lost,player_games_won,player_games_lost
130718,R128,Carlos Alcaraz,Mark Lajal,7-6(3) 7-5 6-2,3.0,0.0,20.0,13.0
130774,R64,Carlos Alcaraz,Aleksandar Vukic,7-6(5) 6-2 6-2,3.0,0.0,19.0,10.0
130802,R32,Carlos Alcaraz,Frances Tiafoe,5-7 6-2 4-6 7-6(2) 6-2,3.0,2.0,28.0,23.0
130816,R16,Carlos Alcaraz,Ugo Humbert,6-3 6-4 1-6 7-5,3.0,1.0,20.0,18.0
130823,QF,Carlos Alcaraz,Tommy Paul,5-7 6-4 6-2 6-2,3.0,1.0,23.0,15.0
130826,SF,Carlos Alcaraz,Daniil Medvedev,6-7(1) 6-3 6-4 6-4,3.0,1.0,24.0,18.0


Totals: {'player_sets_won': 18.0, 'player_sets_lost': 5.0, 'player_games_won': 134.0, 'player_games_lost': 97.0}

Novak Djokovic — previous rounds


,round,winner_name,loser_name,score,player_sets_won,player_sets_lost,player_games_won,player_games_lost
130765,R128,Novak Djokovic,Vit Kopriva,6-1 6-2 6-2,3.0,0.0,18.0,5.0
130797,R64,Novak Djokovic,Jacob Fearnley,6-3 6-4 5-7 7-5,3.0,1.0,24.0,19.0
130813,R32,Novak Djokovic,Alexei Popyrin,4-6 6-3 6-4 7-6(3),3.0,1.0,23.0,19.0
130821,R16,Novak Djokovic,Holger Rune,6-3 6-4 6-2,3.0,0.0,18.0,9.0
130827,SF,Novak Djokovic,Lorenzo Musetti,6-4 7-6(2) 6-4,3.0,0.0,19.0,14.0


Totals: {'player_sets_won': 15.0, 'player_sets_lost': 2.0, 'player_games_won': 102.0, 'player_games_lost': 66.0}


In [78]:
base_numeric_features = [
    # Match context
    "draw_size",
    "best_of",
    "round_order",

    # Ranking / physical
    "p1_seed",
    "p2_seed",
    "p1_is_seeded",
    "p2_is_seeded",
    "seed_advantage",
    "p1_age",
    "p2_age",
    "age_difference",
    "p1_ht",
    "p2_ht",
    "height_difference",
    "p1_rank",
    "p2_rank",
    "rank_difference",
    "p1_rank_points",
    "p2_rank_points",
    "rank_points_difference",

    # General Elo
    "p1_elo",
    "p2_elo",
    "elo_difference",
    "elo_total",
    "p1_matches_played",
    "p2_matches_played",
    "experience_difference",

    # Surface Elo
    "p1_surface_elo",
    "p2_surface_elo",
    "surface_elo_difference",
    "surface_elo_total",
    "p1_surface_matches_played",
    "p2_surface_matches_played",
    "surface_experience_difference",
]


categorical_features = [
    "surface",
    "tourney_level",
    "round",
    "hand_matchup",
    "p1_entry",
    "p2_entry",
]


engineered_numeric_features = (
    rolling_feature_names
    + h2h_feature_names
    + tournament_feature_names
    + tournament_comparison_feature_names
    + tournament_score_feature_names
    + tournament_score_comparison_names
)


numeric_features = list(
    dict.fromkeys(
        base_numeric_features
        + engineered_numeric_features
    )
)

model_feature_columns = (
    numeric_features
    + categorical_features
)


metadata_columns = [
    "source_year",
    "tourney_date",
    "tourney_id",
    "tourney_name",
    "match_num",
    "p1_name",
    "p2_name",
    "target",
]


model_data = state_matches.loc[
    state_matches["is_model_eligible"],
    metadata_columns + model_feature_columns,
].copy()


print("Model rows:", len(model_data))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(model_feature_columns))

Model rows: 120917
Numeric features: 113
Categorical features: 6
Total features: 119


In [79]:
forbidden_feature_terms = (
    "winner",
    "loser",
    "postmatch",
    "score",
    "target",
    "match_status",
    "winner_is_p1",
)


suspicious_features = [
    feature
    for feature in model_feature_columns
    if any(
        forbidden_term in feature
        for forbidden_term in forbidden_feature_terms
    )
]


assert not suspicious_features, (
    f"Potential leakage features: {suspicious_features}"
)

assert model_data["target"].isin([0, 1]).all()

assert model_data["tourney_date"].is_monotonic_increasing

assert len(model_feature_columns) == len(
    set(model_feature_columns)
)


print("Explicit feature registry is leakage-safe")

Explicit feature registry is leakage-safe


In [80]:
missing_summary = pd.DataFrame(
    {
        "missing_count": model_data[
            model_feature_columns
        ].isna().sum(),

        "missing_percent": (
            model_data[
                model_feature_columns
            ]
            .isna()
            .mean()
            .mul(100)
        ),

        "dtype": model_data[
            model_feature_columns
        ].dtypes,
    }
).sort_values(
    "missing_percent",
    ascending=False,
)


missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
]


display(
    missing_summary.round(
        {"missing_percent": 2}
    )
)

,missing_count,missing_percent,dtype
seed_advantage,109576,90.62,float64
p2_entry,100435,83.06,str
p1_entry,100421,83.05,str
p2_seed,79526,65.77,float64
p1_seed,79137,65.45,float64
rank_points_difference,26560,21.97,float64
p1_rank_points,26269,21.72,float64
p2_rank_points,26251,21.71,float64
rank_difference,6775,5.60,float64
p2_rank,4335,3.59,float64


In [81]:
for player in ("p1", "p2"):
    model_data[f"{player}_seed"] = (
        model_data[f"{player}_seed"]
        .fillna(model_data["draw_size"] + 1)
    )

    model_data[f"{player}_entry"] = (
        model_data[f"{player}_entry"]
        .fillna("DA")
    )


model_data["seed_advantage"] = (
    model_data["p2_seed"]
    - model_data["p1_seed"]
)


rank_points_missing_by_year = (
    model_data
    .groupby("source_year")[
        ["p1_rank_points", "p2_rank_points"]
    ]
    .apply(lambda frame: frame.isna().mean() * 100)
    .round(2)
)


print(
    model_data[
        [
            "p1_seed",
            "p2_seed",
            "seed_advantage",
            "p1_entry",
            "p2_entry",
        ]
    ].isna().sum()
)

display(rank_points_missing_by_year)

p1_seed           0
p2_seed           0
seed_advantage    0
p1_entry          0
p2_entry          0
dtype: int64


,p1_rank_points,p2_rank_points
source_year,,
1981,100.00,100.00
1982,100.00,100.00
1983,100.00,100.00
1984,100.00,100.00
1985,100.00,100.00
1986,100.00,100.00
1987,100.00,100.00
1988,100.00,100.00
1989,100.00,100.00


In [82]:
model_data_v1 = model_data[
    model_data["source_year"].between(
        1990,
        2025,
    )
].copy()


live_2026_data = model_data[
    model_data["source_year"].eq(2026)
].copy()


train_cv_data = model_data_v1[
    model_data_v1["source_year"] <= 2022
].copy()


test_data = model_data_v1[
    model_data_v1["source_year"].between(
        2023,
        2025,
    )
].copy()


assert (
    train_cv_data["tourney_date"].max()
    < test_data["tourney_date"].min()
)


print("Train + CV:", train_cv_data.shape)
print("Final test:", test_data.shape)
print("2026 live holdout:", live_2026_data.shape)

print(
    "Train/CV dates:",
    train_cv_data["tourney_date"].min(),
    "→",
    train_cv_data["tourney_date"].max(),
)

print(
    "Test dates:",
    test_data["tourney_date"].min(),
    "→",
    test_data["tourney_date"].max(),
)

Train + CV: (86728, 127)
Final test: (7892, 127)
2026 live holdout: (1278, 127)
Train/CV dates: 1990-01-01 00:00:00 → 2022-11-14 00:00:00
Test dates: 2023-01-02 00:00:00 → 2025-12-17 00:00:00


In [83]:
split_summary = pd.DataFrame(
    {
        "train_cv": {
            "rows": len(train_cv_data),
            "target_mean": train_cv_data["target"].mean(),
            "start": train_cv_data["tourney_date"].min(),
            "end": train_cv_data["tourney_date"].max(),
        },
        "test": {
            "rows": len(test_data),
            "target_mean": test_data["target"].mean(),
            "start": test_data["tourney_date"].min(),
            "end": test_data["tourney_date"].max(),
        },
        "live_2026": {
            "rows": len(live_2026_data),
            "target_mean": live_2026_data["target"].mean(),
            "start": live_2026_data["tourney_date"].min(),
            "end": live_2026_data["tourney_date"].max(),
        },
    }
).T

display(split_summary)

assert train_cv_data["target"].between(0, 1).all()
assert test_data["target"].between(0, 1).all()
assert live_2026_data["target"].between(0, 1).all()

print("Final temporal split checks passed")

,rows,target_mean,start,end
train_cv,86728,0.500334,1990-01-01 00:00:00,2022-11-14 00:00:00
test,7892,0.501394,2023-01-02 00:00:00,2025-12-17 00:00:00
live_2026,1278,0.498435,2026-01-04 00:00:00,2026-05-25 00:00:00


Final temporal split checks passed


In [84]:
import json
from pathlib import Path


project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

processed_data_directory = (
    project_root / "data" / "processed"
)

processed_data_directory.mkdir(
    parents=True,
    exist_ok=True,
)


model_export_data = (
    pd.concat(
        [
            model_data_v1,
            live_2026_data,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "tourney_date",
            "tourney_id",
            "match_num",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


dataset_path = (
    processed_data_directory
    / "atp_model_data_1990_2026.pkl"
)

feature_registry_path = (
    processed_data_directory
    / "feature_registry.json"
)


model_export_data.to_pickle(dataset_path)


feature_registry = {
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "model_feature_columns": model_feature_columns,
    "target": "target",
    "training_start_year": 1990,
    "train_cv_end_year": 2022,
    "test_start_year": 2023,
    "test_end_year": 2025,
    "live_holdout_year": 2026,
}

feature_registry_path.write_text(
    json.dumps(
        feature_registry,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("Dataset saved:", dataset_path)
print("Feature registry saved:", feature_registry_path)
print("Export shape:", model_export_data.shape)

Dataset saved: data/processed/atp_model_data_1990_2026.pkl
Feature registry saved: data/processed/feature_registry.json
Export shape: (95898, 127)
